
# CANS Risk Profile Analysis

Cleaned notebook version with:
- clearer section flow
- reusable helper functions
- default matplotlib colors for general charts
- custom colors used only for the four profiles


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

DATA_PATH = "cleaned_data.csv"  # update this path if needed


## 1. Load data and keep first assessment only

In [ ]:

cans = pd.read_csv(DATA_PATH, parse_dates=["DateCompleted"])

first = (
    cans.sort_values(["OptionsNumber", "DateCompleted"])
        .groupby("OptionsNumber", as_index=False)
        .first()
)

print("rows:", cans.shape)
print("unique youth in raw data:", cans["OptionsNumber"].nunique())
print("rows after first assessment filter:", first.shape)
print("unique youth after filter:", first["OptionsNumber"].nunique())


## 2. Define CANS domains

In [ ]:

be_cols = [
    "Psychosis",
    "Impulsivity/Hyperactivity",
    "Depression",
    "Anxiety",
    "Oppositional",
    "Conduct",
    "Anger Control",
    "Substance Use",
    "Adjustment To Trauma",
]

lf_cols = [
    "Family Function",
    "Living Situation",
    "Social Functioning",
    "Developmental/Intellectual",
    "Decision Making/Judgment",
    "School Behavior",
    "School Achievement",
    "School Attendance",
    "Medical/Physical Health",
    "Sexual Development",
    "Sleep",
]

rb_cols = [
    "Suicide Risk",
    "Non-Suicidal Self Injurious Behavior",
    "Other Self Harm (Recklessness)",
    "Danger To Others",
    "Sexual Aggression",
    "Delinquent Behavior",
    "Runaway",
    "Intentional Misbehavior/Social Behavior",
]

first["be_mean"] = first[be_cols].mean(axis=1)
first["lf_mean"] = first[lf_cols].mean(axis=1)
first["rb_mean"] = first[rb_cols].mean(axis=1)

first[["be_mean", "lf_mean", "rb_mean"]].head()


## 3. Cluster youth into four profiles

In [ ]:

X = first[["be_mean", "lf_mean", "rb_mean"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=50)
first["cluster"] = kmeans.fit_predict(X_scaled)
first["cluster"].value_counts().sort_index()


In [ ]:

cluster_map = {
    0: "Profile A",  # Behavioral + Risk
    1: "Profile D",  # Low Acuity
    2: "Profile B",  # Functioning-Driven
    3: "Profile C",  # High Acuity
}

profile_order = ["Profile A", "Profile B", "Profile C", "Profile D"]
profile_labels = {
    "Profile A": "Behavioral + Risk",
    "Profile B": "Functioning-Driven",
    "Profile C": "High Acuity",
    "Profile D": "Low Acuity",
}
profile_colors = {
    "Profile A": "#D84B4B",
    "Profile B": "#E39B43",
    "Profile C": "#7A6ACB",
    "Profile D": "#55A868",
}

first["profile"] = first["cluster"].map(cluster_map)
first["profile"] = pd.Categorical(first["profile"], categories=profile_order, ordered=True)


## 4. Overview tables

In [ ]:

summary = (
    first.groupby("profile", observed=True)
        .agg(
            n=("OptionsNumber", "size"),
            avg_age=("AgeWhenAssessed", "mean"),
            be=("be_mean", "mean"),
            lf=("lf_mean", "mean"),
            rb=("rb_mean", "mean"),
        )
        .reset_index()
)

summary["avg_age"] = summary["avg_age"].round(1)
summary[["be", "lf", "rb"]] = summary[["be", "lf", "rb"]].round(3)
summary


In [ ]:

incident_rates = {
    "Profile A": 11.0,
    "Profile B": 9.7,
    "Profile C": 6.2,
    "Profile D": 4.4,
}

overview_table = summary.copy()
overview_table["incident_rate"] = overview_table["profile"].astype(str).map(incident_rates)
overview_table


## 5. Overview charts

In [ ]:

# Profile size (default color)
plt.figure(figsize=(6.5, 4))
plt.bar(overview_table["profile"].astype(str), overview_table["n"])
plt.xlabel("Profile")
plt.ylabel("Number of Youth")
plt.title("Profile Size")
plt.show()


In [ ]:

# Domain scores by profile (only profile charts use custom colors)
domains = ["Behavioral/Emotional", "Life Functioning", "Risk Behaviors"]
x = np.arange(len(domains))
width = 0.2

plt.figure(figsize=(8, 4.5))
for i, profile in enumerate(profile_order):
    row = overview_table.loc[overview_table["profile"] == profile, ["be", "lf", "rb"]]
    vals = row.values.flatten()
    plt.bar(x + (i - 1.5) * width, vals, width=width, label=profile, color=profile_colors[profile])

plt.xticks(x, domains)
plt.ylabel("Mean score (0-3)")
plt.title("Domain Scores by Profile")
plt.legend(frameon=False, ncol=2)
plt.show()


In [ ]:

# Incident rate by profile (only profile charts use custom colors)
ir_df = overview_table.copy()

plt.figure(figsize=(6.5, 4))
plt.bar(
    ir_df["profile"].astype(str),
    ir_df["incident_rate"],
    color=[profile_colors[p] for p in ir_df["profile"].astype(str)],
)
plt.ylabel("Incident Rate (%)")
plt.title("Incident Rate by Profile")
plt.show()


In [ ]:

print("Total youth:", 1062)
print("Overall incident rate:", "7.6%")
print("Chi-square p-value:", 0.008)
print("Silhouette score:", 0.393)


## 6. Demographic summaries

In [ ]:

eth_summary = (
    first.groupby("profile", observed=True)
         .agg(
             pct_male=("M", "mean"),
             latino_pct=("Latino", "mean"),
             black_pct=("Black", "mean"),
             asian_pct=("Asian", "mean"),
         )
         .reset_index()
)

eth_summary[["pct_male", "latino_pct", "black_pct", "asian_pct"]] = (
    eth_summary[["pct_male", "latino_pct", "black_pct", "asian_pct"]] * 100
).round(1)

eth_summary


In [ ]:

plt.figure(figsize=(6.5, 4))
plt.bar(eth_summary["profile"].astype(str), eth_summary["pct_male"])
plt.ylabel("% Male")
plt.title("Gender (% Male) by Profile")
plt.ylim(0, 80)
plt.show()


In [ ]:

x = np.arange(len(eth_summary))
width = 0.22

plt.figure(figsize=(7.5, 4.5))
plt.bar(x - width, eth_summary["latino_pct"], width, label="Latino")
plt.bar(x, eth_summary["black_pct"], width, label="Black")
plt.bar(x + width, eth_summary["asian_pct"], width, label="Asian")
plt.xticks(x, eth_summary["profile"].astype(str))
plt.ylabel("Percent")
plt.title("Ethnicity Breakdown by Profile")
plt.ylim(0, 55)
plt.legend(frameon=False)
plt.show()


In [ ]:

age_df = (
    first.groupby(["profile", "AgeWhenAssessed"], observed=True)
         .size()
         .reset_index(name="count")
)

plt.figure(figsize=(10, 4.5))
for profile in profile_order:
    sub = age_df[age_df["profile"] == profile]
    plt.plot(
        sub["AgeWhenAssessed"],
        sub["count"],
        marker="o",
        linewidth=2,
        markersize=4,
        label=profile,
        color=profile_colors[profile],
    )

plt.xlabel("Age")
plt.ylabel("Count")
plt.title("Age Distribution by Profile")
plt.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.15))
plt.show()


## 7. Helper functions for profile deep dive

In [ ]:

def get_profile_deep_dive(profile_name):
    df = first[first["profile"] == profile_name].copy()

    eth_row = (
        eth_summary.loc[eth_summary["profile"] == profile_name]
        .iloc[0]
        .to_dict()
    )

    overview = {
        "profile": profile_name,
        "label": profile_labels[profile_name],
        "youth": len(df),
        "avg_age": round(df["AgeWhenAssessed"].mean(), 1),
        "be": round(df["be_mean"].mean(), 3),
        "lf": round(df["lf_mean"].mean(), 3),
        "rb": round(df["rb_mean"].mean(), 3),
        "pct_male": eth_row["pct_male"],
        "latino_pct": eth_row["latino_pct"],
        "black_pct": eth_row["black_pct"],
        "asian_pct": eth_row["asian_pct"],
        "incident_rate": incident_rates[profile_name],
    }

    def _make_table(cols, domain_name):
        out = (
            df[cols].mean()
              .sort_values(ascending=False)
              .reset_index()
        )
        out.columns = ["item", "mean_score"]
        out["mean_score"] = out["mean_score"].round(2)
        out["domain"] = domain_name
        return out

    be_table = _make_table(be_cols, "Behavioral/Emotional")
    lf_table = _make_table(lf_cols, "Life Functioning")
    rb_table = _make_table(rb_cols, "Risk Behaviors")

    item_table = pd.concat([be_table, lf_table, rb_table], ignore_index=True)
    item_table = item_table.sort_values("mean_score", ascending=False).reset_index(drop=True)

    return overview, be_table, lf_table, rb_table, item_table


def plot_domain_summary(overview, profile_name):
    vals = [overview["be"], overview["lf"], overview["rb"]]
    labels = ["Behavioral/Emotional", "Life Functioning", "Risk Behaviors"]
    plt.figure(figsize=(6, 4))
    plt.bar(labels, vals, color=profile_colors[profile_name])
    plt.ylim(0, 2.5)
    plt.ylabel("Mean score (0-3)")
    plt.title(f"{profile_name}: Domain Summary Scores")
    plt.show()


def plot_item_bars(table, title, profile_name):
    plt.figure(figsize=(7, max(3.5, len(table) * 0.3)))
    plt.barh(table["item"], table["mean_score"], color=profile_colors[profile_name])
    plt.gca().invert_yaxis()
    plt.xlim(0, 3)
    plt.xlabel("Mean score")
    plt.title(title)
    plt.show()


def profile_glance_table(overview):
    return pd.DataFrame({
        "metric": [
            "Youth", "Incident Rate", "Avg Age", "% Male",
            "Latino", "Black", "Asian", "BE Mean", "LF Mean", "RB Mean"
        ],
        "value": [
            overview["youth"],
            f"{overview['incident_rate']}%",
            overview["avg_age"],
            f"{overview['pct_male']}%",
            f"{overview['latino_pct']}%",
            f"{overview['black_pct']}%",
            f"{overview['asian_pct']}%",
            overview["be"],
            overview["lf"],
            overview["rb"],
        ]
    })


## 8. Profile deep dive

### Profile A — Behavioral + Risk

In [ ]:

overview, be_table, lf_table, rb_table, item_table = get_profile_deep_dive("Profile A")
profile_glance_table(overview)


In [ ]:

plot_domain_summary(overview, overview["profile"])
plot_item_bars(be_table, f"{overview['profile']}: Behavioral / Emotional", overview["profile"])
plot_item_bars(lf_table, f"{overview['profile']}: Life Functioning", overview["profile"])
plot_item_bars(rb_table, f"{overview['profile']}: Risk Behaviors", overview["profile"])


In [ ]:
item_table

### Profile B — Functioning-Driven

In [ ]:

overview, be_table, lf_table, rb_table, item_table = get_profile_deep_dive("Profile B")
profile_glance_table(overview)


In [ ]:

plot_domain_summary(overview, overview["profile"])
plot_item_bars(be_table, f"{overview['profile']}: Behavioral / Emotional", overview["profile"])
plot_item_bars(lf_table, f"{overview['profile']}: Life Functioning", overview["profile"])
plot_item_bars(rb_table, f"{overview['profile']}: Risk Behaviors", overview["profile"])


In [ ]:
item_table

### Profile C — High Acuity

In [ ]:

overview, be_table, lf_table, rb_table, item_table = get_profile_deep_dive("Profile C")
profile_glance_table(overview)


In [ ]:

plot_domain_summary(overview, overview["profile"])
plot_item_bars(be_table, f"{overview['profile']}: Behavioral / Emotional", overview["profile"])
plot_item_bars(lf_table, f"{overview['profile']}: Life Functioning", overview["profile"])
plot_item_bars(rb_table, f"{overview['profile']}: Risk Behaviors", overview["profile"])


In [ ]:
item_table

### Profile D — Low Acuity

In [ ]:

overview, be_table, lf_table, rb_table, item_table = get_profile_deep_dive("Profile D")
profile_glance_table(overview)


In [ ]:

plot_domain_summary(overview, overview["profile"])
plot_item_bars(be_table, f"{overview['profile']}: Behavioral / Emotional", overview["profile"])
plot_item_bars(lf_table, f"{overview['profile']}: Life Functioning", overview["profile"])
plot_item_bars(rb_table, f"{overview['profile']}: Risk Behaviors", overview["profile"])


In [ ]:
item_table

## 9. Key findings


**Finding 1 — Risk behaviors are the strongest incident predictor**

Profile A has moderate behavioral and life-functioning scores, but elevated risk behavior scores. It has the highest incident rate.



**Finding 2 — Functional impairment alone elevates risk**

Profile B has lower risk behavior scores than Profile A, but still shows a high incident rate. This suggests that functional breakdown can matter even when acute risk items are not the most elevated.



**Finding 3 — The acuity paradox: most severe does not automatically mean highest incident risk**

Profile C scores highest across domains, but its incident rate is lower than Profiles A and B.



**Finding 4 — Low acuity is the largest and lowest-risk group**

Profile D is the largest profile and has the lowest incident rate, which makes it a useful baseline group.
